<a href="https://colab.research.google.com/github/briliananugra/mobile-legends-sentiment-analysis/blob/main/mobile_legends_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/briliananugra/mobile-legends-sentiment-analysis/main/data/ml_sample_raw.csv"
df_sample = pd.read_csv(url)

print(df_sample.shape)
df_sample.head()

(3000, 3)


,content,score,at
0,woy monton bisa buat game gak udah empat belas...,1,2026-07-27 07:57:30
1,KALAU NGASIH TEAM YG BENER LAHHH,1,2026-07-27 07:54:47
2,mooton bujang intinya game pp game bujang main...,1,2026-07-27 07:53:01
3,capek juga main selalu kalah saya sebagai solo...,1,2026-07-27 07:52:40
4,"Pengen balik ke ml tolong, jangan tambahkan fi...",2,2026-07-27 07:52:24


---
## 📌 Catatan: Sel Eksplorasi Awal (One-Time)

Sel-sel di bawah ini (distribusi score, cek data kosong/duplikat, contoh teks, dan simpan CSV) adalah **eksplorasi awal Checkpoint 1** yang sudah dilakukan sekali dan hasilnya sudah didokumentasikan.

- **Tidak perlu dijalankan ulang** setiap buka notebook — cukup dijalankan kalau ingin verifikasi ulang data.
- Data hasil eksplorasi ini sudah tersimpan permanen di GitHub: `data/ml_sample_raw.csv`
- Sel ini dipertahankan sebagai dokumentasi proses penelitian (bukti kerja analitis), bukan langkah wajib di setiap sesi.
---

In [ ]:
# distribusi rating
print("Distribusi score:")
print(df_sample['score'].value_counts().sort_index())

# cek kosong & duplikat
print("\nData kosong per kolom:")
print(df_sample.isnull().sum())
print("\nJumlah duplikat (content):", df_sample['content'].duplicated().sum())

# contoh teks untuk cek kualitas (bahasa gaul, emoji, dll)
print("\nContoh 5 ulasan:")
for i, t in enumerate(df_sample['content'].sample(5, random_state=42)):
    print(f"{i+1}. {t}\n")

Distribusi score:
score
1    1695
2     170
3     136
4     141
5     858
Name: count, dtype: int64

Data kosong per kolom:
content    0
score      0
at         0
dtype: int64

Jumlah duplikat (content): 154

Contoh 5 ulasan:
1. setiap proses pertandingan kalo tim musuh atau tim sendiri pake skin animasi ga bisa masuk loading nya lama udah gitu dihitung afk

2. mantap

3. game nya gak enak , di kasih tim bot semua di kasih kalah terus naik kagak yang ada malah buang buang waktu aja

4. keren

5. menurut ku game ini bagus tapi tambah bnyak skin lagi bagus sihh



> ⚠️ Sel ini tidak perlu dijalankan ulang — data sudah tersimpan di GitHub (`data/ml_sample_raw.csv`) dan tidak berubah.

In [ ]:
df_sample.to_csv('/content/ml_sample_raw.csv', index=False)
print("Tersimpan: /content/ml_sample_raw.csv")

Tersimpan: /content/ml_sample_raw.csv


# Checkpoint 2 — Tahap Preprocessing

In [ ]:
!pip install -q Sastrawi emoji

import pandas as pd

url = "https://raw.githubusercontent.com/briliananugra/mobile-legends-sentiment-analysis/main/data/ml_sample_raw.csv"
df = pd.read_csv(url)
print("Sebelum drop duplikat:", df.shape)

df = df.drop_duplicates(subset="content").reset_index(drop=True)
print("Setelah drop duplikat:", df.shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 22.6 MB/s eta 0:00:00
Sebelum drop duplikat: (3000, 3)
Setelah drop duplikat: (2846, 3)


In [ ]:
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
import emoji
import unicodedata

# --- peta huruf small caps (ᴡᴀʟᴀᴜᴘᴜɴ dsb) -> huruf latin biasa; tidak tertangani oleh NFKD ---
SMALLCAPS_MAP = {
    "ᴀ":"a","ʙ":"b","ᴄ":"c","ᴅ":"d","ᴇ":"e","ꜰ":"f","ɢ":"g","ʜ":"h","ɪ":"i",
    "ᴊ":"j","ᴋ":"k","ʟ":"l","ᴍ":"m","ɴ":"n","ᴏ":"o","ᴘ":"p","ǫ":"q","ʀ":"r",
    "ѕ":"s","ᴛ":"t","ᴜ":"u","ᴠ":"v","ᴡ":"w","x":"x","ʏ":"y","ᴢ":"z",
}

# --- kamus emoji -> tag sentimen kasar ---
EMOJI_SENTIMENT = {
    "👍": "emoji_positif", "🥰": "emoji_positif", "😍": "emoji_positif",
    "❤️": "emoji_positif", "💜": "emoji_positif", "😊": "emoji_positif",
    "🤩": "emoji_positif", "😁": "emoji_positif", "👏": "emoji_positif",
    "💯": "emoji_positif", "😘": "emoji_positif", "✨": "emoji_positif",
    "😇": "emoji_positif", "🎉": "emoji_positif", "😄": "emoji_positif",
    "🌟": "emoji_positif",
    "😭": "emoji_negatif", "😡": "emoji_negatif", "🤬": "emoji_negatif",
    "👎": "emoji_negatif", "😤": "emoji_negatif", "🥲": "emoji_negatif",
    "😢": "emoji_negatif", "💢": "emoji_negatif", "🤢": "emoji_negatif",
    "😞": "emoji_negatif", "😔": "emoji_negatif", "💔": "emoji_negatif",
    "😩": "emoji_negatif", "😠": "emoji_negatif", "🙄": "emoji_negatif",
    "👹": "emoji_negatif", "👺": "emoji_negatif",
    "🗿": "emoji_netral", "😹": "emoji_netral", "🤣": "emoji_netral",
    "😂": "emoji_netral", "🤡": "emoji_netral", "💀": "emoji_netral",
    "😅": "emoji_netral", "🙏": "emoji_netral",
}

def extract_emoji_tags(text):
    """Keluarkan semua emoji jadi tag sentimen, kembalikan (teks_tanpa_emoji, list_tag)."""
    tags = []
    for e, tag in EMOJI_SENTIMENT.items():
        count = text.count(e)
        if count:
            tags.extend([tag] * count)
            text = text.replace(e, " ")
    remaining = emoji.emoji_list(text)          # emoji lain yang tidak ada di kamus
    tags.extend(["emoji_lain"] * len(remaining))
    text = emoji.replace_emoji(text, replace=" ")
    return text, tags

# --- kamus normalisasi kata gaul / tidak baku (termasuk istilah khas ML) ---
SLANG_DICT = {
    # variasi typo "moonton" (nama publisher, banyak sekali variannya di data)
    "monton": "moonton", "muntun": "moonton", "montoon": "moonton",
    "montol": "moonton", "montod": "moonton", "montool": "moonton",
    "montoll": "moonton", "monoton": "moonton", "mootoon": "moonton",
    "mooton": "moonton", "muntoon": "moonton",
    # kata ganti & partikel informal
    "gw": "saya", "gue": "saya", "gua": "saya", "aq": "saya", "ak": "saya",
    "lu": "kamu", "lo": "kamu", "km": "kamu",
    "yg": "yang", "dr": "dari", "tp": "tapi", "dgn": "dengan", "sm": "sama",
    "dpt": "dapat", "jd": "jadi", "blm": "belum", "udh": "sudah", "dah": "sudah",
    "kl": "kalau", "klo": "kalau", "kalo": "kalau", "gmn": "bagaimana",
    "knp": "kenapa", "sy": "saya", "bgt": "banget", "bngt": "banget",
    "gk": "tidak", "ga": "tidak", "nggak": "tidak", "engga": "tidak",
    "enggak": "tidak", "gak": "tidak", "kaga": "tidak",
    "jgn": "jangan", "trs": "terus", "utk": "untuk", "krn": "karena",
    "karna": "karena", "bs": "bisa", "biar": "supaya",
    # istilah khas komunitas Mobile Legends
    "ws": "winstreak", "ls": "losestreak", "wr": "winrate",
    "cit": "cheater", "citer": "cheater",
    "mm": "matchmaking", "hp": "handphone",
}

def normalize_slang(text):
    words = text.split()
    return " ".join(SLANG_DICT.get(w, w) for w in words)

def clean_text(text):
    text = re.sub(r"http\S+|www\.\S+", " ", text)      # URL
    text = unicodedata.normalize("NFKD", text)           # ubah huruf gaya (bold script, dsb) -> huruf latin biasa
    text = "".join(SMALLCAPS_MAP.get(ch, ch) for ch in text)  # tangani small caps (ᴡᴀʟᴀᴜᴘᴜɴ dsb) yang tidak tertangani NFKD
    text = re.sub(r"\d+", " ", text)                    # angka
    text = re.sub(r"[^a-z\s_]", " ", text)               # selain huruf latin -> spasi
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)            # huruf berulang berlebih
    text = re.sub(r"\s+", " ", text).strip()
    return text

# --- stopword removal (kata negasi TIDAK dihapus agar makna sentimen tidak berbalik) ---
stopword_factory = StopWordRemoverFactory()
stopwords = set(stopword_factory.get_stop_words())
NEGATION_KEEP = {"tidak", "bukan", "jangan", "kurang", "tanpa", "belum"}
stopwords = stopwords - NEGATION_KEEP

def remove_stopwords(text):
    return " ".join(w for w in text.split() if w not in stopwords)

# --- stemmer ---
stemmer = StemmerFactory().create_stemmer()

def full_pipeline(text, use_stemming=False):
    text = text.lower()
    text, emoji_tags = extract_emoji_tags(text)
    text = clean_text(text)
    text = normalize_slang(text)
    text = remove_stopwords(text)
    if use_stemming:
        text = stemmer.stem(text)
    text = re.sub(r"\s+", " ", text).strip()
    if emoji_tags:
        text = (text + " " + " ".join(emoji_tags)).strip()
    return text

print("Fungsi preprocessing siap dipakai.")

Fungsi preprocessing siap dipakai.


In [ ]:
df["clean_no_stem"] = df["content"].apply(lambda t: full_pipeline(t, use_stemming=False))
df["clean_stem"]    = df["content"].apply(lambda t: full_pipeline(t, use_stemming=True))

df[["content", "clean_no_stem", "clean_stem"]].head(10)

,content,clean_no_stem,clean_stem
0,woy monton bisa buat game gak udah empat belas...,woy moonton buat game tidak udah empat belas k...,woy moonton buat game tidak udah empat belas k...
1,KALAU NGASIH TEAM YG BENER LAHHH,kalau ngasih team bener lahh,kalau ngasih team bener lahh
2,mooton bujang intinya game pp game bujang main...,moonton bujang intinya game pp game bujang mai...,moonton bujang inti game pp game bujang main k...
3,capek juga main selalu kalah saya sebagai solo...,capek main selalu kalah solo rankk tim selalu ...,capek main selalu kalah solo rankk tim selalu ...
4,"Pengen balik ke ml tolong, jangan tambahkan fi...",pengen balik ml jangan tambahkan fitur fitur b...,ken balik ml jangan tambah fitur fitur bikin n...
5,game sistem gj kasih aja gw kalahh terus tim m...,game sistem gj kasih aja kalahh terus tim musu...,game sistem gj kasih aja kalahh terus tim musu...
6,game gateli lose trike tros,game gateli lose trike tros,game gateli lose trike tros
7,makasih moontoon,makasih moontoon,makasih moontoon
8,Sumpah ini moonton kejam dah.Lagi asik main ti...,sumpah moonton kejam asik main tiba malah rest...,sumpah moonton kejam asik main tiba malah rest...
9,bagus keren,bagus keren,bagus keren


In [ ]:
vocab_no_stem = set(" ".join(df["clean_no_stem"]).split())
vocab_stem    = set(" ".join(df["clean_stem"]).split())

avg_len_no_stem = df["clean_no_stem"].apply(lambda t: len(t.split())).mean()
avg_len_stem    = df["clean_stem"].apply(lambda t: len(t.split())).mean()

print("=== Tanpa stemming ===")
print("Ukuran vocab :", len(vocab_no_stem))
print("Rata-rata token/ulasan:", round(avg_len_no_stem, 2))

print("\n=== Dengan stemming ===")
print("Ukuran vocab :", len(vocab_stem))
print("Rata-rata token/ulasan:", round(avg_len_stem, 2))

print("\nSelisih vocab (tereduksi oleh stemming):", len(vocab_no_stem) - len(vocab_stem))

=== Tanpa stemming ===
Ukuran vocab : 4899
Rata-rata token/ulasan: 14.36

=== Dengan stemming ===
Ukuran vocab : 4079
Rata-rata token/ulasan: 14.36

Selisih vocab (tereduksi oleh stemming): 820


In [ ]:
kosong_no_stem = (df["clean_no_stem"].str.strip() == "").sum()
kosong_stem    = (df["clean_stem"].str.strip() == "").sum()
print("Baris kosong (no stem):", kosong_no_stem)
print("Baris kosong (stem)   :", kosong_stem)

df[df["clean_no_stem"].str.strip() == ""][["content", "score"]]

Baris kosong (no stem): 1
Baris kosong (stem)   : 1


,content,score
342,ok,5


In [ ]:
# drop baris yang hasil cleaning-nya kosong (hanya berlaku untuk clean_no_stem;
# baris yang sama juga kosong di clean_stem karena stemming tidak bikin teks jadi ada isi dari kosong)
df = df[df["clean_no_stem"].str.strip() != ""].reset_index(drop=True)
print("Jumlah baris setelah drop baris kosong:", len(df))

Jumlah baris setelah drop baris kosong: 2845


In [ ]:
df.to_csv("/content/ml_sample_preprocessed.csv", index=False)
print("Tersimpan: /content/ml_sample_preprocessed.csv")
print("Kolom:", list(df.columns))
print("Jumlah baris final:", len(df))

Tersimpan: /content/ml_sample_preprocessed.csv
Kolom: ['content', 'score', 'at', 'clean_no_stem', 'clean_stem']
Jumlah baris final: 2845


# Checkpoint 3 — Pelabelan Sentimen & Split Data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/briliananugra/mobile-legends-sentiment-analysis/main/data/ml_sample_preprocessed.csv"
df = pd.read_csv(url)
print("Jumlah baris:", len(df))
print("Kolom:", list(df.columns))

# InSet Lexicon (Koto & Rahmaningtyas, 2017) — 3.609 kata positif, 6.609 kata negatif, bobot -5 s.d. +5
!wget -q "https://raw.githubusercontent.com/fajri91/InSet/master/positive.tsv" -O positive.tsv
!wget -q "https://raw.githubusercontent.com/fajri91/InSet/master/negative.tsv" -O negative.tsv

pos = pd.read_csv("positive.tsv", sep="\t")
neg = pd.read_csv("negative.tsv", sep="\t")
print("Kata positif:", len(pos), "| Kata negatif:", len(neg))

Jumlah baris: 2845
Kolom: ['content', 'score', 'at', 'clean_no_stem', 'clean_stem']
Kata positif: 3609 | Kata negatif: 6609


In [ ]:
# --- bangun base lexicon dari InSet (jumlahkan bobot jika kata muncul di kedua file) ---
base_lexicon = {}
for _, r in pos.iterrows():
    w = str(r["word"]).strip()
    if " " not in w:          # hanya kata tunggal (skip frasa multi-kata untuk kesederhanaan)
        base_lexicon[w] = base_lexicon.get(w, 0) + float(r["weight"])
for _, r in neg.iterrows():
    w = str(r["word"]).strip()
    if " " not in w:
        base_lexicon[w] = base_lexicon.get(w, 0) + float(r["weight"])

print("Ukuran base lexicon (kata tunggal):", len(base_lexicon))

# --- kamus tambahan (opsional, bisa dimatikan dengan USE_DOMAIN_LEXICON = False) ---
DOMAIN_LEXICON_ADDITIONS = {
    # kata yang net weight InSet-nya berlawanan dengan makna umum di ulasan game
    "bagus": 3, "menang": 3, "seru": 3,
    # istilah umum ulasan game yang OOV (tidak ada sama sekali di InSet)
    "keren": 4, "mantul": 4, "gg": 3, "enak": 3, "kece": 3,
    "ngelag": -3, "lag": -3, "toxic": -4, "cheater": -4, "afk": -2,
    "bug": -2, "error": -2, "troll": -3,
}
full_lexicon = {**base_lexicon, **DOMAIN_LEXICON_ADDITIONS}

# --- kata negasi (sama seperti yang dipertahankan saat stopword removal di Checkpoint 2) ---
NEGATION_WORDS = {"tidak", "bukan", "jangan", "kurang", "tanpa", "belum"}

def score_text(text, lexicon):
    """Jumlahkan bobot kata yang cocok; balik tanda jika didahului kata negasi."""
    tokens = str(text).split()
    score = 0.0
    for i, tok in enumerate(tokens):
        if tok in lexicon:
            w = lexicon[tok]
            if i > 0 and tokens[i-1] in NEGATION_WORDS:
                w = -w
            score += w
    return score

def label_from_score(score, rating):
    """rating (kolom score asli 1-5) dipakai HANYA sebagai fallback saat skor lexicon 0 (tidak ada
    kata sentimen yang terdeteksi sama sekali) — bukan sebagai sumber label utama."""
    if score > 0:
        return "Positif"
    elif score < 0:
        return "Negatif"
    else:
        return "Positif" if rating >= 4 else "Negatif"

print("Fungsi scoring siap dipakai.")

Ukuran base lexicon (kata tunggal): 8395
Fungsi scoring siap dipakai.


In [ ]:
df["skor_base"] = df["clean_no_stem"].apply(lambda t: score_text(t, base_lexicon))
df["label_base"] = df.apply(lambda r: label_from_score(r["skor_base"], r["score"]), axis=1)

df["skor_domain"] = df["clean_no_stem"].apply(lambda t: score_text(t, full_lexicon))
df["label_domain"] = df.apply(lambda r: label_from_score(r["skor_domain"], r["score"]), axis=1)

print("=== Distribusi label (InSet murni) ===")
print(df["label_base"].value_counts())
print("\n=== Distribusi label (InSet + kamus domain) ===")
print(df["label_domain"].value_counts())

=== Distribusi label (InSet murni) ===
label_base
Negatif    2060
Positif     785
Name: count, dtype: int64

=== Distribusi label (InSet + kamus domain) ===
label_domain
Negatif    1939
Positif     906
Name: count, dtype: int64


In [ ]:
# label referensi kasar dari rating bintang (skor 3 dikecualikan karena ambigu)
df_ref = df[df["score"] != 3].copy()
df_ref["label_rating"] = df_ref["score"].apply(lambda s: "Positif" if s >= 4 else "Negatif")

agree_base = (df_ref["label_base"] == df_ref["label_rating"]).mean()
agree_domain = (df_ref["label_domain"] == df_ref["label_rating"]).mean()

print(f"Baris yang dibandingkan (rating != 3): {len(df_ref)}")
print(f"Kesesuaian label InSet murni vs rating   : {agree_base:.2%}")
print(f"Kesesuaian label InSet+domain vs rating  : {agree_domain:.2%}")

Baris yang dibandingkan (rating != 3): 2711
Kesesuaian label InSet murni vs rating   : 74.99%
Kesesuaian label InSet+domain vs rating  : 78.90%


In [ ]:
beda = df[df["label_base"] != df["label_domain"]][["content", "clean_no_stem", "score", "label_base", "label_domain"]]
print(f"Jumlah ulasan yang labelnya berubah karena kamus domain: {len(beda)}")
beda.sample(min(10, len(beda)), random_state=42)

Jumlah ulasan yang labelnya berubah karena kamus domain: 207


,content,clean_no_stem,score,label_base,label_domain
2239,GAME RUSAK ORANG RELOG PAS LAGI IN GAME GA BIS...,game rusak orang relog pas in game tidak masuk...,1,Negatif,Positif
150,bagus,bagus,4,Negatif,Positif
1026,permainannya sangat bagus,permainannya sangat bagus,5,Negatif,Positif
1314,sangatlah bagus,sangatlah bagus,5,Negatif,Positif
2302,game nge bug,game nge bug,1,Positif,Negatif
108,toxic,toxic,5,Positif,Negatif
1344,tolong ringankan sedikit untuk hp potato karen...,ringankan sedikit handphone potato minggu kema...,5,Negatif,Positif
1897,game bagus playernya tolong di hapus,game bagus playernya hapus,1,Negatif,Positif
196,game Nye bagus dan seru monton plisss beri say...,game nye bagus seru moonton pliss beri winstre...,1,Negatif,Positif
2073,halo. kenapa aku stuck di loding ml dan tidak ...,halo aku stuck loding ml tidak masuk game pada...,1,Negatif,Positif


In [ ]:
# rekomendasi: pakai label_domain sebagai label final (lebih sesuai konteks ulasan game),
# tapi label_base tetap disimpan di CSV untuk transparansi/pembanding
df["label"] = df["label_domain"]

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
df["split"] = "train"
df.loc[test_df.index, "split"] = "test"

print("Distribusi label final:")
print(df["label"].value_counts())
print("\nJumlah data train:", len(train_df))
print("Jumlah data test :", len(test_df))
print("\nDistribusi label di train:")
print(train_df["label"].value_counts(normalize=True).round(3))
print("\nDistribusi label di test:")
print(test_df["label"].value_counts(normalize=True).round(3))

Distribusi label final:
label
Negatif    1939
Positif     906
Name: count, dtype: int64

Jumlah data train: 2276
Jumlah data test : 569

Distribusi label di train:
label
Negatif    0.681
Positif    0.319
Name: proportion, dtype: float64

Distribusi label di test:
label
Negatif    0.682
Positif    0.318
Name: proportion, dtype: float64


In [ ]:
df.to_csv("/content/ml_sample_labeled.csv", index=False)
print("Tersimpan: /content/ml_sample_labeled.csv")
print("Kolom:", list(df.columns))

Tersimpan: /content/ml_sample_labeled.csv
Kolom: ['content', 'score', 'at', 'clean_no_stem', 'clean_stem', 'skor_base', 'label_base', 'skor_domain', 'label_domain', 'label', 'split']


## Melihat data Tertentu

In [ ]:
row = df.loc[2073]
print("content      :", row["content"])
print("clean_no_stem:", row["clean_no_stem"])
print("score (rating):", row["score"])
print("skor_domain  :", row["skor_domain"])
print("label_domain :", row["label_domain"])

# bonus: tunjukkan kata per kata yang match ke lexicon + bobotnya
for i, tok in enumerate(row["clean_no_stem"].split()):
    if tok in full_lexicon:
        w = full_lexicon[tok]
        prev = row["clean_no_stem"].split()[i-1] if i > 0 else None
        flipped = prev in NEGATION_WORDS if prev else False
        print(f"  token='{tok}' bobot={w} {'(DIBALIK karena negasi)' if flipped else ''}")

content      : halo. kenapa aku stuck di loding ml dan tidak bisa masuk ke game padahal siyal ku bagus bagus aja tetapi setelah aku relock beberapa kali sudah bisa. tapi nanti terulang lagi dan karakter loby ku kaku gak bisa bergerak cuman bisa muncul teck pembicaraan dan juga lagi aku sering ketemu bot/computer saat main apakah memang sudah terlalu sedikit player yang main?
clean_no_stem: halo aku stuck loding ml tidak masuk game padahal siyal ku bagus bagus aja aku relock beberapa kali terulang karakter loby ku kaku tidak bergerak cuman muncul teck pembicaraan aku sering ketemu bot computer main memang terlalu sedikit player main
score (rating): 1
skor_domain  : 1.0
label_domain : Positif
  token='halo' bobot=1.0 
  token='aku' bobot=1.0 
  token='tidak' bobot=-5.0 
  token='masuk' bobot=-3.0 (DIBALIK karena negasi)
  token='game' bobot=2.0 
  token='bagus' bobot=3 
  token='bagus' bobot=3 
  token='aja' bobot=1.0 
  token='aku' bobot=1.0 
  token='tidak' bobot=-5.0 
  token='muncul'

# Checkpoint 4 — Ekstraksi Fitur & Modeling

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                              confusion_matrix, classification_report)

!pip install -q imbalanced-learn
from imblearn.over_sampling import SMOTE

url = "https://raw.githubusercontent.com/briliananugra/mobile-legends-sentiment-analysis/main/data/ml_sample_labeled.csv"
df = pd.read_csv(url)

train_df = df[df["split"] == "train"].copy()
test_df  = df[df["split"] == "test"].copy()

print("Data train:", len(train_df), "| Data test:", len(test_df))

Data train: 2276 | Data test: 569


In [3]:
summary_rows = []
saved_models = {}   # simpan model/prediksi tiap kombinasi untuk dipakai lagi di Cell 4

def evaluate(name, text_col, use_smote, model_type):
    vec = TfidfVectorizer()
    X_train = vec.fit_transform(train_df[text_col].fillna(""))
    X_test  = vec.transform(test_df[text_col].fillna(""))
    y_train = train_df["label"]
    y_test  = test_df["label"]

    if use_smote:
        sm = SMOTE(random_state=42)
        X_train, y_train = sm.fit_resample(X_train, y_train)

    model = MultinomialNB() if model_type == "NB" else SVC(kernel="linear", C=1.0, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    acc = accuracy_score(y_test, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, pred, average="macro")
    f1_pos  = precision_recall_fscore_support(y_test, pred, labels=["Positif"], average=None)[2][0]
    f1_neg  = precision_recall_fscore_support(y_test, pred, labels=["Negatif"], average=None)[2][0]
    rec_pos = precision_recall_fscore_support(y_test, pred, labels=["Positif"], average=None)[1][0]

    summary_rows.append({
        "Model": name, "Teks": text_col, "SMOTE": "Ya" if use_smote else "Tidak",
        "Accuracy": round(acc, 4), "F1_macro": round(f1, 4),
        "Recall_Positif": round(rec_pos, 4), "F1_Positif": round(f1_pos, 4), "F1_Negatif": round(f1_neg, 4)
    })
    saved_models[(name, text_col, use_smote)] = (model, vec, pred, y_test)
    return model, vec, pred

print("Fungsi evaluate() siap dipakai.")

Fungsi evaluate() siap dipakai.


In [4]:
configs = [
    ("Naive Bayes",  "clean_no_stem", False, "NB"),
    ("Naive Bayes",  "clean_stem",    False, "NB"),
    ("Naive Bayes",  "clean_no_stem", True,  "NB"),
    ("Naive Bayes",  "clean_stem",    True,  "NB"),
    ("SVM (linear)", "clean_no_stem", False, "SVM"),
    ("SVM (linear)", "clean_stem",    False, "SVM"),
    ("SVM (linear)", "clean_no_stem", True,  "SVM"),
    ("SVM (linear)", "clean_stem",    True,  "SVM"),
]
for name, col, smote, mtype in configs:
    evaluate(name, col, smote, mtype)

comparison_df = pd.DataFrame(summary_rows).sort_values("F1_macro", ascending=False)
print(comparison_df.to_string(index=False))
comparison_df.to_csv("/content/comparison_table.csv", index=False)

       Model          Teks SMOTE  Accuracy  F1_macro  Recall_Positif  F1_Positif  F1_Negatif
SVM (linear)    clean_stem    Ya    0.8576    0.8376          0.7956      0.7805      0.8947
SVM (linear)    clean_stem Tidak    0.8612    0.8340          0.7182      0.7670      0.9011
SVM (linear) clean_no_stem    Ya    0.8489    0.8253          0.7569      0.7611      0.8895
SVM (linear) clean_no_stem Tidak    0.8506    0.8183          0.6740      0.7416      0.8949
 Naive Bayes clean_no_stem    Ya    0.8190    0.7836          0.6519      0.6962      0.8711
 Naive Bayes    clean_stem    Ya    0.8155    0.7816          0.6630      0.6957      0.8676
 Naive Bayes    clean_stem Tidak    0.7627    0.6447          0.2928      0.4398      0.8495
 Naive Bayes clean_no_stem Tidak    0.7504    0.6180          0.2541      0.3932      0.8429


# Checkpoint 5 — Evaluasi & Analisis Hasil

In [6]:
best = comparison_df.iloc[0]
key = (best["Model"], best["Teks"], best["SMOTE"] == "Ya")
best_model, best_vec, best_pred, y_test = saved_models[key]

print("=== Model terbaik:", key, "===")
print(classification_report(y_test, best_pred, digits=4))

cm = confusion_matrix(y_test, best_pred, labels=["Negatif", "Positif"])
print("Confusion Matrix [baris=aktual, kolom=prediksi], urutan [Negatif, Positif]:")
print(cm)

=== Model terbaik: ('SVM (linear)', 'clean_stem', True) ===
              precision    recall  f1-score   support

     Negatif     0.9029    0.8866    0.8947       388
     Positif     0.7660    0.7956    0.7805       181

    accuracy                         0.8576       569
   macro avg     0.8344    0.8411    0.8376       569
weighted avg     0.8593    0.8576    0.8583       569

Confusion Matrix [baris=aktual, kolom=prediksi], urutan [Negatif, Positif]:
[[344  44]
 [ 37 144]]


In [7]:
test_out = test_df.copy()
test_out["pred_best"] = best_pred
test_out["benar"] = test_out["label"] == test_out["pred_best"]
test_out.to_csv("/content/predictions_best_model.csv", index=False)

print("=== Contoh BENAR diklasifikasi ===")
print(test_out[test_out["benar"]][["content", "label"]].sample(5, random_state=1))

print("\n=== Contoh SALAH diklasifikasi ===")
print(test_out[~test_out["benar"]][["content", "label", "pred_best"]].sample(
    min(8, (~test_out["benar"]).sum()), random_state=1))

=== Contoh BENAR diklasifikasi ===
                                                content    label
2093  ngasih tim drafsistem mlu...kalah mah 10 kali ...  Negatif
656          Game ga adil, pembagian tim tidak seimbang  Negatif
950   game buruk, saya memainkan game ini banyak pro...  Negatif
2480  taikk game gak jelas lawan imortal temen epic ...  Negatif
2404  game nya hama sering ngelag pakek kuota apalag...  Negatif

=== Contoh SALAH diklasifikasi ===
                                                content    label pred_best
2371  bikin stres ama ping nya merah mulu giliran ga...  Positif   Negatif
1117  geme nya bagus aku udh main dari lama juga tap...  Negatif   Positif
1298                                     game nya buruk  Negatif   Positif
2556                 aneh sekarang ping nya tinggi mulu  Positif   Negatif
1864                                 i hate you moonton  Negatif   Positif
1943  omon omon katanya mau memperbaiki sistem matcm...  Positif   Negatif
2070  INI GAME